## RFDiffusion3 Pipeline

# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Generation** | RFD3 | Generate novel proteins via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation via Refolding** | RF3 | Predict the structure from designed sequence to validate designability |

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → RF3 (validation) → RMSD comparison
```

---

In [0]:
%restart_python

In [0]:
!foundry list-available

In [0]:
import os
import warnings
# Set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

overarching_save_dir = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/rfdiffusion3"

In [0]:
import yaml
import os
import random
import string 

# Load in the path_input_pdb from the yaml file
path_task_yaml = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/inputs/gopher_alpha_snake.yaml"
with open(path_task_yaml, 'r') as file:
    config_task = yaml.safe_load(file)

path_input_pdb = config_task['path_input_pdb']
# Define a Python dictionary
config_data = {
    'spec_1': {
        'input': path_input_pdb,
        'contig': "C1-193,/0,10-30,A61-67,10-30,D10-12,20-30", # Clash between the 2 motifs so shortening one of them (Original Works on May 10th but inherent motif clash)
        'select_hotspots' : "C9,C10,C11,C12,C13,C14,C46,C47,C48,C70,C72,C73,C74,C75,C76,C77,C78,C79,C112,C113,C114,C116,C143",
        'infer_ori_strategy' : "hotspots",
        'is_non_loopy' : True
  },
    'spec_2': {
        'input': path_input_pdb,
        'contig': "C1-193,/0,30-70,A107-112,10-30,D10-12,20-30", # Works on May 11th Runs so keeping to try again
        'select_hotspots' : "C9,C10,C11,C12,C13,C14,C46,C47,C48,C70,C72,C73,C74,C75,C76,C77,C78,C79,C112,C113,C114,C116,C143",
        'infer_ori_strategy' : "hotspots",
        'is_non_loopy' : True
  }, 
}

# Define the output YAML file path
path_yaml_dir = os.path.join(overarching_save_dir, "yaml")

name = 'biosim_col'
path = name
while os.path.exists(f"{path_yaml_dir}/{path}.yaml"):
  path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

path_yaml = os.path.join(path_yaml_dir, f"{path}.yaml")
# Save the dictionary to a YAML file
with open(path_yaml, 'w') as file:
    yaml.dump(config_data, file, default_flow_style=False)

print(f"YAML configuration saved to {path_yaml}")

In [0]:
import os
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Set seed for reproducibility
seed_everything(0)

In [0]:
import shutil
OUTPUT_DIR = f"{overarching_save_dir}/designs/{path}"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

# RFD3 Design Command Line Arguments
path_output_dir = os.path.abspath(OUTPUT_DIR)
low_memory_mode = False
n_batches = 4
diffusion_batch_size = 7 

!rfd3 design out_dir="{path_output_dir}" inputs="{path_yaml}" prevalidate_inputs=True low_memory_mode={low_memory_mode} n_batches={n_batches} diffusion_batch_size={diffusion_batch_size}

In [0]:
import gzip
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/MotifScaffold/utils")
import biotite.structure.io.pdb as pdb
import biotite.structure.io.pdbx as pdbx
from utils import *
from CreateRFDYaml import CreateRFDYaml

obj_create_rfd_yaml = CreateRFDYaml()

def extract_atom_array_from_gzip(file_path: str, save_pdb_path: str = ""):
    """ Extracts the atom array from a gzipped file containing a PDBx/mmCIF file. Saves to a new destination if save_path is provided

    Args:
        file_path (str): The path to the gzipped file.

    Returns:
        biotite.structure.AtomArray: The extracted atom array.
    """
    with gzip.open(file_path, "rt") as f:
        cif_file = pdbx.CIFFile.read(f)
        atom_array = pdbx.get_structure(cif_file, model=1)
    
    if save_pdb_path != "":
        pdb_file = pdb.PDBFile()
        pdb_file.set_structure(atom_array)
        pdb_file.write(save_pdb_path)
    
    return atom_array

def create_af2_mpnn_yaml(path_mpnn_pdb_input: str, path_mpnn_af2_output: str, contig: str, num_rfd_designs: int, yaml_save_path: str):
    """ Creating a modified version of the typical YAML to run Sergey's AF2 and MPNN pipeline 
        
        Args:
            path_mpnn_pdb_input: refers to path of the input pdb designed via RFDiffusion (must end with _0) as other designs are sampled by replacing _0 with _1, _2 and etc.
            path_mpnn_af2_output: refers to the path of the folder that will contain the outputs from Sergey's MPNN and AF2 pipeline
            contig: refers to the contig string that is used to specify the included (fixed) and designed residues for RFDiffusion & subsequently mpnn
            yaml_save_path: refers to the path of the yaml file that will be created for running Sergey's AF2 and MPNN pipeline

        Additional input parameters saved as new dictionary with key: 'pipeline' and nested keys matching Sergey's inputs

        Returns:
            yaml_save_path: path to the yaml file created for running Sergey's AF2 and MPNN pipeline

    """
    mpnn_dict = obj_create_rfd_yaml.create_mpnn_dict()
    af2_dict = obj_create_rfd_yaml.create_af2_dict()

    assert os.path.exists(path_mpnn_pdb_input), f"Path to input pdb file does not exist: {path_mpnn_pdb_input}"
    assert os.path.exists(path_mpnn_af2_output), f"Path to output directory does not exist: {path_mpnn_af2_output}"

    common_dict = {'pipeline' : {
        'contig_str' : contig, # contig string for RFDiffusion (adjusted formatting for Sergey's pipeline)
        'loc' : path_mpnn_af2_output, # path to save AF2 and MPNN outputs (output_directory path)
        'pdb' : path_mpnn_pdb_input, # path to input pdb file (input_pdb_path) for mpnn (must end with _0)
        'num_designs' : num_rfd_designs, # number of RFD3 designs generated for specific yaml and associated specification (num_designs)
        'copies' : 1 # number of repeating copies (not exactly sure what it means)
        }
    }
    pipeline_dict = common_dict | mpnn_dict | af2_dict

    with open(yaml_save_path, 'w') as file:
        yaml.dump(pipeline_dict, file, default_flow_style=False)
        print(f"MPNN & AF2 YAML configuration saved to {yaml_save_path}")

def contig_rfd3_to_sergey(contig_rfd3: str):
    """ Converts the contig string from RFDiffusion3 format to Sergey's format.

    Args:
        contig_rfd3 (str): The contig string in RFDiffusion3 format.

    Returns:
        contig_sergey (str): The converted contig string in Sergey's format.
    """
    representation_chain_break = "/0"
    contig_target, contig_binder = contig_rfd3.split(f',{representation_chain_break},') 
    contig_binder_sergey = []
    for contig_region in contig_binder.split(','):
    
        # Diffused Linker regions represented by raw integers (no dashes)
        if "-" not in contig_region:
            contig_binder_sergey.append(f"{contig_region}-{contig_region}")
        else:
            contig_binder_sergey.append(contig_region)
    contig_binder_sergey = "/".join(contig_binder_sergey)
    contig_sergey = f"{contig_target}:{contig_binder_sergey}"
    
    return contig_sergey

In [0]:
import yaml
import json

with open(path_yaml, 'r') as file:
    config_rfd3 = yaml.safe_load(file)

print("Path: ", path)
print("Config: ", config_rfd3)

paths_mpnn_af2_yaml = []

for design_id in config_rfd3.keys():
    
    for batch_num in range(0, n_batches):

        # 1. Create folder within path_output_dir for each unique combo of path and design_id (Stores RFD3 designs and Sergey's MPNN & AF2 output)
        folder_name_sergey = f"{path}_{design_id}_{batch_num}"
        path_rfd3_pdb_sergey = os.path.join(path_output_dir, folder_name_sergey)
        os.makedirs(path_rfd3_pdb_sergey, exist_ok= True)

        # 1.5: Extract sampled contig used when running RFD3
        base_contig = config_rfd3[design_id]['contig']
        name_batch_json = f"{folder_name_sergey}_model_0.json"
        path_batch_json = os.path.join(path_output_dir, name_batch_json)
        with open(path_batch_json, 'r') as file:
            batch_json = json.load(file)

        sampled_contig = batch_json['specification']['extra']['sampled_contig']
        sampled_contig_binder = sampled_contig.split('/0,')[1]
        sampled_lengths = iter([x for x in sampled_contig_binder.split(',') if x.isdigit()])
        input_contig_list = base_contig.split(',')
        contig_rfd3 = ','.join((next(sampled_lengths) if x[0].isdigit() else x) for x in (input_contig_list))
        print("RFD3 Sampled Contig: ", contig_rfd3)

        # 2. Find all gzip files in the output directory associated with specific combo of yaml_name and design_id
        names_yaml_design_id_gzip = [file for file in os.listdir(path_output_dir) if folder_name_sergey in file and file.endswith(".gz")]
    
        # 2.5 Make a folder in the path_rfd3_pdb_sergey titled RFDiffusion 3 Designs and copy the RFD3 designs to the folder
        path_rfd3_pdb_sergey_rfd3 = os.path.join(path_rfd3_pdb_sergey, "rfd3_designs")
        os.makedirs(path_rfd3_pdb_sergey_rfd3, exist_ok= True)
        
        for name_yaml_design_id_gzip in names_yaml_design_id_gzip:
            # Move both the gzip files and their associated confidence or json metrics to the folder
            # First move the gzip files. Define old path to file and create new path to file. Then move the file
            path_design_id_gzip = os.path.join(path_output_dir, name_yaml_design_id_gzip)
            path_design_id_gzip_sergey = os.path.join(path_rfd3_pdb_sergey_rfd3, name_yaml_design_id_gzip)
            shutil.move(path_design_id_gzip, path_design_id_gzip_sergey)

            # Now move the json confidence files. Define old path to file and create new path to file. Then move the file
            name_design_json = name_yaml_design_id_gzip.split('.cif')[0] + ".json"
            path_design_id_json = os.path.join(path_output_dir, name_design_json)
            path_design_id_json_sergey = os.path.join(path_rfd3_pdb_sergey_rfd3, name_design_json)
            shutil.move(path_design_id_json, path_design_id_json_sergey)

        # 3. Save the gzip files to pdb files in the folder created in step 1
        for index, name_yaml_design_id_gzip in enumerate(names_yaml_design_id_gzip):
            path_yaml_design_id_gzip = os.path.join(path_rfd3_pdb_sergey_rfd3, name_yaml_design_id_gzip)

            # Create simpler name for pdb file. If ran with n_batches > 1, naming convention involves batch size and model #
            # RFD3 Default naming scheme: {input_yaml_name}_{yaml_rfd3_dict_key}_{batch #}_model_{model #}.cif.gz
            name_design_id_pdb = folder_name_sergey + f"_model_{index}.pdb" 
            print(name_design_id_pdb)
            path_design_id_pdb = os.path.join(path_rfd3_pdb_sergey, name_design_id_pdb)
            _ = extract_atom_array_from_gzip(path_yaml_design_id_gzip, path_design_id_pdb)
    
        # 4. Create the contig string & path_mpnn_input_pdb & path_save_yaml for Sergey's MPNN & AF2 pipeline
        contig_sergey = contig_rfd3_to_sergey(contig_rfd3)
        print("Sergey Contig: ", contig_sergey)
        path_mpnn_input_pdb = "_".join(path_design_id_pdb.split("_")[:-1]) + "_0.pdb"
        path_save_mpnn_af2_yaml = os.path.join(path_rfd3_pdb_sergey, "mpnn_af2.yaml")
        num_rfd_designs = diffusion_batch_size
    
        # 5. Create the yaml file for Sergey's MPNN & AF2 pipeline
        create_af2_mpnn_yaml(path_mpnn_pdb_input= path_mpnn_input_pdb, path_mpnn_af2_output= path_rfd3_pdb_sergey, contig = contig_sergey,
                         num_rfd_designs = num_rfd_designs, yaml_save_path= path_save_mpnn_af2_yaml)
        paths_mpnn_af2_yaml.append(path_save_mpnn_af2_yaml)
    

In [0]:
print(paths_mpnn_af2_yaml)
# Pass over the paths_mpnn_af2_yaml, path to input_pdb, path to output_dir
dbutils.jobs.taskValues.set(key = "paths_mpnn_af2_yaml", value = paths_mpnn_af2_yaml)
dbutils.jobs.taskValues.set(key = "path_task_yaml", value = path_task_yaml)
dbutils.jobs.taskValues.set(key = "path_design_campaign", value = path_output_dir)

In [0]:
import gzip
import shutil
import random
import biotite
import biotite.structure as struc
import biotite.structure.io.pdb as pdb

def load_designed_structure(path_output_dir: str, yaml_name: str, design_id: str, model_id: int = 0):
    """ Loads the designed structure from the specified yaml file and design ID.

    Args:
        path_output_dir (str): The path to the output directory containing the designed structures
        yaml_name (str): The name of the yaml file.
        design_identifier (str): The Identifier of the design.
        model_id (int, optional): The ID of the model. Defaults to 0.

    Returns:
        atom_array (biotite.structure.AtomArray): The loaded atom array.
    """
    path_design_folder = os.path.join(path_output_dir, f"{yaml_name}_{design_id}")
    path_design_pdb = os.path.join(path_design_folder, f"{yaml_name}_{design_id}_0_model_{model_id}.pdb")
    print("Structure Name: ", path_design_pdb.split('/')[-1])
    pdb_file = pdb.PDBFile.read(path_design_pdb)
    atom_array = pdb_file.get_structure(model = 1)
    return atom_array

design_id = list(config_rfd3.keys())[0]
model_id = random.randint(0, num_rfd_designs - 1)
print(f"Loading designed structure for yaml_name: {path} and design_identifier: {design_id} and with model_id: {model_id}")
atom_array = load_designed_structure(path_output_dir = path_output_dir, yaml_name = path, design_id = design_id, model_id = model_id)
view(atom_array)